In [1]:
import pandas as pd
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [2]:
# Project folders
PROJECT_ROOT = Path("..")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
QUALITY_REPORT_PATH = PROJECT_ROOT / "data" / "quality_reports"

# Create reports folder if it doesn't exist
QUALITY_REPORT_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
csv_files = list(RAW_DATA_PATH.glob("*.csv"))

tables = {}

for file in csv_files:
    table_name = file.stem
    tables[table_name] = pd.read_csv(file)

print(f"Loaded {len(tables)} tables.")

Loaded 12 tables.


In [4]:
for table_name in tables:
    print(table_name)

categories
customers
employees
orders
order_items
payments
products
promotions
returns
shipments
stores
suppliers


In [5]:
print("=" * 70)
print("SUPPLYVISION DATA QUALITY FRAMEWORK")
print("=" * 70)

quality_dimensions = [
    "Completeness",
    "Uniqueness",
    "Referential Integrity",
    "Validity",
    "Consistency"
]

for i, dimension in enumerate(quality_dimensions, start=1):
    print(f"{i}. {dimension}")

SUPPLYVISION DATA QUALITY FRAMEWORK
1. Completeness
2. Uniqueness
3. Referential Integrity
4. Validity
5. Consistency


Completeness Check

Business Purpose

Completeness measures whether all required data is present. Missing values can lead to inaccurate KPIs, failed joins, incorrect aggregations, and unreliable business insights.

This validation checks every table for missing values and calculates the percentage of missing data for each column.

In [6]:
def check_missing_values(df, table_name):
    """
    Check missing values for a table.
    Returns a summary DataFrame.
    """

    missing_count = df.isna().sum()
    missing_percent = (missing_count / len(df)) * 100

    report = pd.DataFrame({
        "Table": table_name,
        "Column": missing_count.index,
        "Missing Values": missing_count.values,
        "Missing %": missing_percent.round(2).values
    })

    return report

In [7]:
missing_reports = []

for table_name, df in tables.items():
    report = check_missing_values(df, table_name)
    missing_reports.append(report)

missing_report = pd.concat(missing_reports, ignore_index=True)

missing_report.head()

,Table,Column,Missing Values,Missing %
0,categories,category_id,0,0.0
1,categories,category_name,0,0.0
2,customers,customer_id,0,0.0
3,customers,city,0,0.0
4,customers,signup_date,0,0.0


In [8]:
print("="*70)
print("MISSING VALUE SUMMARY")
print("="*70)

total_missing = missing_report["Missing Values"].sum()

print(f"Total Missing Values : {total_missing}")

MISSING VALUE SUMMARY
Total Missing Values : 0


In [9]:
missing_report.to_csv(
    QUALITY_REPORT_PATH / "missing_value_report.csv",
    index=False
)

print("Missing Value Report saved successfully.")

Missing Value Report saved successfully.


## Uniqueness Check

### Business Purpose

Primary keys uniquely identify every record in a table.

Duplicate primary keys can cause:

- Incorrect joins
- Duplicate transactions
- Double-counted revenue
- Data integrity issues

This validation ensures that every primary key remains unique before loading the data into the database.

In [10]:
primary_keys = {
    "categories": "category_id",
    "customers": "customer_id",
    "employees": "employee_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "payments": "payment_id",
    "products": "product_id",
    "promotions": "promotion_id",
    "returns": "return_id",
    "shipments": "shipment_id",
    "stores": "store_id",
    "suppliers": "supplier_id"
}

In [11]:
def check_primary_key_uniqueness(df, table_name, primary_key):
    """
    Validate uniqueness of the primary key.
    """

    duplicate_count = df.duplicated(subset=primary_key).sum()

    return {
        "Table": table_name,
        "Primary Key": primary_key,
        "Duplicate Records": duplicate_count,
        "Status": "PASS" if duplicate_count == 0 else "FAIL"
    }

In [12]:
pk_results = []

for table_name, primary_key in primary_keys.items():

    result = check_primary_key_uniqueness(
        tables[table_name],
        table_name,
        primary_key
    )

    pk_results.append(result)

pk_report = pd.DataFrame(pk_results)

pk_report

,Table,Primary Key,Duplicate Records,Status
0,categories,category_id,0,PASS
1,customers,customer_id,0,PASS
2,employees,employee_id,0,PASS
3,orders,order_id,0,PASS
4,order_items,order_item_id,0,PASS
5,payments,payment_id,0,PASS
6,products,product_id,0,PASS
7,promotions,promotion_id,0,PASS
8,returns,return_id,0,PASS
9,shipments,shipment_id,0,PASS


In [13]:
pk_report.to_csv(
    QUALITY_REPORT_PATH / "primary_key_report.csv",
    index=False
)

print("Primary Key Report saved successfully.")

Primary Key Report saved successfully.


## Referential Integrity Check

### Business Purpose

Referential integrity ensures that every foreign key in a child table references a valid primary key in the corresponding parent table.

Broken relationships can lead to:

- Missing records after joins
- Incorrect KPIs
- Inaccurate reporting
- Data integrity issues

This validation checks every foreign key relationship defined in the retail database.

In [14]:
foreign_keys = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "store_id", "stores", "store_id"),
    ("orders", "promotion_id", "promotions", "promotion_id"),

    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),

    ("products", "category_id", "categories", "category_id"),
    ("products", "supplier_id", "suppliers", "supplier_id"),

    ("payments", "order_id", "orders", "order_id"),

    ("shipments", "order_id", "orders", "order_id"),

    ("returns", "order_item_id", "order_items", "order_item_id")
]

In [15]:
def check_foreign_key(child_df,
                      parent_df,
                      child_table,
                      parent_table,
                      child_key,
                      parent_key):
    """
    Validate foreign key relationships.
    """

    invalid_records = (
        ~child_df[child_key].isin(parent_df[parent_key])
    ).sum()

    return {
        "Child Table": child_table,
        "Foreign Key": child_key,
        "Parent Table": parent_table,
        "Parent Key": parent_key,
        "Invalid Records": invalid_records,
        "Status": "PASS" if invalid_records == 0 else "FAIL"
    }

In [16]:
fk_results = []

for child_table, child_key, parent_table, parent_key in foreign_keys:

    result = check_foreign_key(
        tables[child_table],
        tables[parent_table],
        child_table,
        parent_table,
        child_key,
        parent_key
    )

    fk_results.append(result)

fk_report = pd.DataFrame(fk_results)

fk_report

,Child Table,Foreign Key,Parent Table,Parent Key,Invalid Records,Status
0,orders,customer_id,customers,customer_id,0,PASS
1,orders,store_id,stores,store_id,0,PASS
2,orders,promotion_id,promotions,promotion_id,0,PASS
3,order_items,order_id,orders,order_id,0,PASS
4,order_items,product_id,products,product_id,0,PASS
5,products,category_id,categories,category_id,0,PASS
6,products,supplier_id,suppliers,supplier_id,0,PASS
7,payments,order_id,orders,order_id,0,PASS
8,shipments,order_id,orders,order_id,0,PASS
9,returns,order_item_id,order_items,order_item_id,0,PASS


In [17]:
fk_report.to_csv(
    QUALITY_REPORT_PATH / "foreign_key_report.csv",
    index=False
)

print("Foreign Key Report saved successfully.")

Foreign Key Report saved successfully.


## Validity Check

### Business Purpose

Validity ensures that values conform to expected business rules.

Even when relationships are correct, invalid values such as negative prices, impossible discounts, or unexpected shipment statuses can lead to misleading business insights.

This validation checks numeric ranges and allowed categorical values across the dataset.

In [18]:
validity_rules = {
    "products": {
        "price": lambda x: x > 0
    },

    "order_items": {
        "qty": lambda x: x > 0,
        "price": lambda x: x > 0
    },

    "payments": {
        "amount": lambda x: x > 0
    },

    "promotions": {
        "discount": lambda x: (x >= 0) & (x <= 100)
    },

    "returns": {
        "refund": lambda x: x >= 0
    },

    "employees": {
        "salary": lambda x: x > 0
    },

    "shipments": {
        "status": lambda x: x.isin([
            "delivered",
            "shipped",
            "late"
        ])
    }
}

In [19]:
def check_validity(df, table_name, rules):
    """
    Validate business rules for each column.
    """

    results = []

    for column, rule in rules.items():

        invalid_count = (~rule(df[column])).sum()

        results.append({
            "Table": table_name,
            "Column": column,
            "Invalid Records": invalid_count,
            "Status": "PASS" if invalid_count == 0 else "FAIL"
        })

    return results

In [20]:
validity_results = []

for table_name, rules in validity_rules.items():

    validity_results.extend(
        check_validity(
            tables[table_name],
            table_name,
            rules
        )
    )

validity_report = pd.DataFrame(validity_results)

validity_report

,Table,Column,Invalid Records,Status
0,products,price,0,PASS
1,order_items,qty,0,PASS
2,order_items,price,0,PASS
3,payments,amount,0,PASS
4,promotions,discount,0,PASS
5,returns,refund,0,PASS
6,employees,salary,0,PASS
7,shipments,status,0,PASS


## Data Quality Summary

### Business Purpose

This section consolidates all validation results into a single summary that provides an overall view of dataset quality.

The summary will later be used for:

- ETL monitoring
- Data Governance
- Power BI KPI Cards
- Executive reporting|

In [21]:
summary = pd.DataFrame({
    "Quality Dimension": [
        "Completeness",
        "Primary Key Uniqueness",
        "Referential Integrity",
        "Business Validity"
    ],

    "Checks": [
        len(missing_report),
        len(pk_report),
        len(fk_report),
        len(validity_report)
    ],

    "Failures": [
        (missing_report["Missing Values"] > 0).sum(),
        (pk_report["Duplicate Records"] > 0).sum(),
        (fk_report["Invalid Records"] > 0).sum(),
        (validity_report["Invalid Records"] > 0).sum()
    ]
})

In [22]:
summary["Passed"] = summary["Checks"] - summary["Failures"]

summary["Status"] = summary["Failures"].apply(
    lambda x: "PASS" if x == 0 else "FAIL"
)

summary

,Quality Dimension,Checks,Failures,Passed,Status
0,Completeness,37,0,37,PASS
1,Primary Key Uniqueness,12,0,12,PASS
2,Referential Integrity,10,0,10,PASS
3,Business Validity,8,0,8,PASS


In [23]:
total_checks = summary["Checks"].sum()

total_failures = summary["Failures"].sum()

quality_score = (
    (total_checks - total_failures)
    / total_checks
) * 100

print(f"Overall Data Quality Score : {quality_score:.2f}%")

Overall Data Quality Score : 100.00%


In [24]:
summary.to_csv(
    QUALITY_REPORT_PATH / "quality_summary.csv",
    index=False
)

print("Quality Summary saved successfully.")

Quality Summary saved successfully.


In [25]:
from datetime import datetime

In [26]:
total_rows = sum(len(df) for df in tables.values())

print("=" * 70)
print("SUPPLYVISION ETL VALIDATION LOG")
print("=" * 70)

print(f"Execution Time      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Tables Checked      : {len(tables)}")
print(f"Rows Checked        : {total_rows:,}")
print(f"Missing Values      : {missing_report['Missing Values'].sum()}")
print(f"Duplicate PKs       : {pk_report['Duplicate Records'].sum()}")
print(f"Broken FK Records   : {fk_report['Invalid Records'].sum()}")
print(f"Invalid Values      : {validity_report['Invalid Records'].sum()}")
print(f"Overall Status      : {'PASS' if total_failures == 0 else 'FAIL'}")
print(f"Quality Score       : {quality_score:.2f}%")

print("=" * 70)

SUPPLYVISION ETL VALIDATION LOG
Execution Time      : 2026-07-30 09:39:05
Tables Checked      : 12
Rows Checked        : 1,591,380
Missing Values      : 0
Duplicate PKs       : 0
Broken FK Records   : 0
Invalid Values      : 0
Overall Status      : PASS
Quality Score       : 100.00%


In [27]:
log_file = QUALITY_REPORT_PATH / "etl_validation_log.txt"

with open(log_file, "w") as f:
    f.write("=" * 70 + "\n")
    f.write("SUPPLYVISION ETL VALIDATION LOG\n")
    f.write("=" * 70 + "\n")
    f.write(f"Execution Time      : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Tables Checked      : {len(tables)}\n")
    f.write(f"Rows Checked        : {total_rows:,}\n")
    f.write(f"Missing Values      : {missing_report['Missing Values'].sum()}\n")
    f.write(f"Duplicate PKs       : {pk_report['Duplicate Records'].sum()}\n")
    f.write(f"Broken FK Records   : {fk_report['Invalid Records'].sum()}\n")
    f.write(f"Invalid Values      : {validity_report['Invalid Records'].sum()}\n")
    f.write(f"Overall Status      : {'PASS' if total_failures == 0 else 'FAIL'}\n")
    f.write(f"Quality Score       : {quality_score:.2f}%\n")
    f.write("=" * 70 + "\n")

print("ETL Validation Log saved successfully.")

ETL Validation Log saved successfully.


## Key Findings

- All 12 tables were successfully validated.
- No missing values were detected.
- All primary keys are unique.
- All foreign key relationships are valid.
- All business validation rules passed.
- Overall Data Quality Score: 100%.

**Conclusion:** The dataset is suitable for ETL processing and downstream analytics.